# Jev vs fine-tuned ModernBERT - parent_queue routing

Head-to-head on the fixed OpsPilot test split (3,563 tickets, `random_state=42`), 7-class `clean_v1` taxonomy.

- **ModernBERT**: `shubhamjoshipro/opspilot-routing-modernbert-base-clean-v1`, fine-tuned on 16,622 examples
- **Jev**: zero-shot, no training data

Runtime > Change runtime type > **T4 GPU** before running.

## 1. Install and mount

Colab ships torch; we only need `transformers`. Drive is mounted to reach the
fixed splits already uploaded to `My Drive/opspilot/`.

In [ ]:
get_ipython().system('pip install -q transformers')

import os, io, json, time, zipfile, threading, random, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
WORK = Path('/content/jev_benchmark')
WORK.mkdir(exist_ok=True)
print('mounted')

## 2. Load the fixed test split

Searches Drive for `ticket_splits.zip` (uploaded from the local repo) and
extracts `test.csv`. Using the saved split matters: it is the exact set the
published ModernBERT numbers were measured on.

In [ ]:
def find_split() -> Path:
    root = Path('/content/drive/MyDrive')
    direct = list(root.rglob('test.csv'))
    for candidate in direct:
        if 'opspilot' in str(candidate).lower() or 'ticket' in str(candidate).lower():
            return candidate
    for archive in root.rglob('ticket_splits.zip'):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORK)
        found = list(WORK.rglob('test.csv'))
        if found:
            return found[0]
    if direct:
        return direct[0]
    raise FileNotFoundError('No test.csv or ticket_splits.zip found under MyDrive')

TEST_PATH = find_split()
df = pd.read_csv(TEST_PATH)
print(f'{TEST_PATH}  ->  {len(df)} rows')

LABEL_COL = 'category' if 'category' in df.columns else 'true_category'
ID_COL = 'ticket_id' if 'ticket_id' in df.columns else 'external_id'

def build_text(row) -> str:
    subject = str(row.get('subject') or '').strip()
    body = str(row.get('body') or '').strip()
    if not subject and not body:
        return str(row.get('customer_message') or '').strip()
    return f'{subject}\n\n{body}'.strip()

df['_text'] = [build_text(r) for _, r in df.iterrows()]
print(df[LABEL_COL].value_counts())

## 3. API key

Typed here rather than stored in the notebook, so the key never lands in the
.ipynb file or in Drive.

In [ ]:
from getpass import getpass

TYPESAFE_API_KEY = getpass('TypeSafe API key: ').strip()
assert TYPESAFE_API_KEY and not TYPESAFE_API_KEY.startswith('http'), 'That looks like a URL, not a key'
print(f'key accepted ({len(TYPESAFE_API_KEY)} chars)')

## 4. Jev (zero-shot)

One Choice question per ticket over the 7 `clean_v1` queues. Criteria are
written to separate the queues that the ModernBERT confusion matrix showed
collapsing into each other.

Set `LIMIT = 50` for a smoke test; `None` runs the full split.

In [ ]:
API_URL = 'https://api.typesafe.ai/v1/systemone'
MODEL = 'jev-latest'
LIMIT = 50
CONCURRENCY = 8

CRITERIA = {
    'technical_product_support': (
        'Anything about the product working, being used, or being accessed: defects '
        'and errors, how-to and configuration questions, integrations, and account '
        'access or environment setup.'),
    'customer_general': (
        'Account and relationship matters with no technical fault, plus generic '
        'questions that fit no other queue: complaints, cancellations, contact '
        'changes, chasing prior tickets, broad information requests.'),
    'billing_and_payments': (
        'Money owed or charged: invoices, payment failures, duplicate or incorrect '
        'charges, refunds, subscription and pricing changes, tax and billing details.'),
    'returns_and_exchanges': (
        'Physical goods moving back or being swapped: returns, exchanges, RMAs, '
        'wrong or damaged items received, shipping a replacement.'),
    'service_outages_and_maintenance': (
        'A shared service is down or degraded for many users, or scheduled '
        'maintenance is the subject. Distinct from one customer isolated fault.'),
    'sales_and_pre_sales': (
        'A prospective or expanding purchase: quotes, pricing for new business, '
        'demos, trials, capability questions asked before buying, upgrades.'),
    'human_resources': (
        'Employment matters: recruitment and applications, onboarding, payroll and '
        'benefits, internal staff or workplace policy questions.'),
}
LABELS = list(CRITERIA)

INSTRUCTIONS = (
    'A customer has written to a support desk. Decide which single operational '
    'queue should own this ticket, based only on the subject and body. Judge what '
    'the customer actually needs done, not the vocabulary they happen to use.')

CLEAN_V1_MAP = {
    'technical_support': 'technical_product_support',
    'it_support': 'technical_product_support',
    'product_support': 'technical_product_support',
    'customer_service': 'customer_general',
    'general_inquiry': 'customer_general',
}
to_clean_v1 = lambda label: CLEAN_V1_MAP.get(label, label)


def ask_jev(state, max_retries=6):
    payload = {'state': state, 'model': MODEL, 'questions': {'queue': {
        'type': 'choice', 'instructions': INSTRUCTIONS, 'criteria': CRITERIA}}}
    body = json.dumps(payload).encode()
    for attempt in range(max_retries):
        req = urllib.request.Request(API_URL, data=body, method='POST', headers={
            'Authorization': f'Bearer {TYPESAFE_API_KEY}',
            'Content-Type': 'application/json'})
        started = time.perf_counter()
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                latency = (time.perf_counter() - started) * 1000
                data = json.loads(resp.read().decode())
            answer = data['answers']['queue']
            answer['_usage'] = data.get('usage', {})
            return answer, latency
        except urllib.error.HTTPError as exc:
            if exc.code in (429, 529) or exc.code >= 500:
                if attempt == max_retries - 1:
                    raise
                time.sleep(min(2 ** attempt, 30) + random.random())
                continue
            raise RuntimeError(f'HTTP {exc.code}: {exc.read().decode()[:300]}')
        except (urllib.error.URLError, TimeoutError):
            if attempt == max_retries - 1:
                raise
            time.sleep(min(2 ** attempt, 30) + random.random())


subset = df if LIMIT is None else df.head(LIMIT)
lock = threading.Lock()
jev_rows, stats = [], {'n': 0, 'correct': 0, 'failed': 0}
started_all = time.perf_counter()

def run_one(item):
    _, row = item
    true_label = to_clean_v1(str(row[LABEL_COL]))
    try:
        answer, latency = ask_jev(row['_text'])
    except Exception as exc:
        with lock:
            stats['failed'] += 1
            print(f'  ! {exc}')
        return
    probs = answer.get('probabilities', {}) or {}
    ordered = sorted(probs.values(), reverse=True)
    top1 = float(ordered[0]) if ordered else 0.0
    top2 = float(ordered[1]) if len(ordered) > 1 else 0.0
    predicted = answer.get('choice', '')
    record = {'ticket_id': str(row[ID_COL]), 'true_label': true_label,
              'predicted_label': predicted, 'confidence': answer.get('confidence', 0.0),
              'top_1_probability': top1, 'top_2_probability': top2,
              'top1_top2_margin': top1 - top2, 'correct': int(predicted == true_label),
              'latency_ms': round(latency, 1),
              'input_tokens': answer.get('_usage', {}).get('input_tokens', 0)}
    for label in LABELS:
        record[f'prob_{label}'] = float(probs.get(label, 0.0))
    with lock:
        jev_rows.append(record)
        stats['n'] += 1
        stats['correct'] += record['correct']
        if stats['n'] % 25 == 0:
            print(f"  {stats['n']}/{len(subset)}  acc={stats['correct']/stats['n']:.4f}")

with ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    list(pool.map(run_one, subset.iterrows()))

jev = pd.DataFrame(jev_rows)
elapsed = time.perf_counter() - started_all
tokens = jev['input_tokens'].sum()
print(f"\nJev: {len(jev)} tickets, acc={jev['correct'].mean():.4f}, failed={stats['failed']}")
print(f"{elapsed:.1f}s wall, median {jev['latency_ms'].median():.0f}ms/ticket")
print(f"{tokens:,} input tokens = ${tokens/1e6*0.042:.4f}")

## 5. Fine-tuned ModernBERT

Pulls the Hub checkpoint and runs it on the identical rows. The Hub repo has no
Inference Provider attached, so the weights run locally on the Colab GPU.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HF_MODEL = 'shubhamjoshipro/opspilot-routing-modernbert-base-clean-v1'
BATCH_SIZE = 32
MAX_LENGTH = 512

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
if device == 'cpu':
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU for a faster run.')

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL).to(device).eval()
id2label = model.config.id2label
mb_labels = [id2label[i] for i in sorted(id2label)]
print(f'{len(mb_labels)} labels: {mb_labels}')
assert set(mb_labels) == set(LABELS), 'Label space differs from the Jev run'

# Same rows Jev saw, in the same order.
mb_subset = subset.loc[subset[ID_COL].astype(str).isin(set(jev['ticket_id']))]
texts = mb_subset['_text'].tolist()
mb_rows = []
started_all = time.perf_counter()

for start in range(0, len(texts), BATCH_SIZE):
    batch = texts[start:start + BATCH_SIZE]
    encoded = tokenizer(batch, truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors='pt').to(device)
    batch_started = time.perf_counter()
    with torch.no_grad():
        logits = model(**encoded).logits
    probs = torch.softmax(logits, dim=-1).cpu()
    per_item_ms = (time.perf_counter() - batch_started) * 1000 / len(batch)

    for offset, row_probs in enumerate(probs):
        idx = start + offset
        source = mb_subset.iloc[idx]
        true_label = to_clean_v1(str(source[LABEL_COL]))
        ordered, _ = torch.sort(row_probs, descending=True)
        top1, top2 = float(ordered[0]), float(ordered[1])
        predicted = mb_labels[int(torch.argmax(row_probs))]
        record = {'ticket_id': str(source[ID_COL]), 'true_label': true_label,
                  'predicted_label': predicted, 'confidence': top1,
                  'top_1_probability': top1, 'top_2_probability': top2,
                  'top1_top2_margin': top1 - top2,
                  'correct': int(predicted == true_label),
                  'latency_ms': round(per_item_ms, 2)}
        for label_idx, label in enumerate(mb_labels):
            record[f'prob_{label}'] = float(row_probs[label_idx])
        mb_rows.append(record)

mb = pd.DataFrame(mb_rows)
elapsed = time.perf_counter() - started_all
print(f"\nModernBERT: {len(mb)} tickets, acc={mb['correct'].mean():.4f}")
print(f'{elapsed:.1f}s wall, {len(mb)/elapsed:.1f} tickets/s on {device}')

## 6. Compare

Accuracy is the headline, but the decisive number is accuracy at a fixed human
review budget: given that a person can only look at N% of tickets, which system
routes the rest more accurately?

McNemar tests whether the accuracy gap is real or sampling noise.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import binomtest

paired = jev.merge(mb, on='ticket_id', suffixes=('_jev', '_mb'))
assert (paired['true_label_jev'] == paired['true_label_mb']).all(), 'label mismatch'
print(f'paired on {len(paired)} tickets\n')

def summarize(frame, name):
    return {'system': name,
            'accuracy': round(accuracy_score(frame['true_label'], frame['predicted_label']), 4),
            'macro_f1': round(f1_score(frame['true_label'], frame['predicted_label'],
                                       average='macro', zero_division=0), 4),
            'weighted_f1': round(f1_score(frame['true_label'], frame['predicted_label'],
                                          average='weighted', zero_division=0), 4),
            'median_latency_ms': round(float(frame['latency_ms'].median()), 1)}

summary = pd.DataFrame([summarize(jev, 'Jev (zero-shot)'),
                        summarize(mb, 'ModernBERT (fine-tuned)')])
print(summary.to_string(index=False))

# McNemar: only the disagreements carry information.
jev_only = int(((paired['correct_jev'] == 1) & (paired['correct_mb'] == 0)).sum())
mb_only = int(((paired['correct_jev'] == 0) & (paired['correct_mb'] == 1)).sum())
print(f'\nJev right / ModernBERT wrong: {jev_only}')
print(f'ModernBERT right / Jev wrong: {mb_only}')
if jev_only + mb_only:
    p = binomtest(jev_only, jev_only + mb_only, 0.5).pvalue
    verdict = 'significant at p<0.05' if p < 0.05 else 'NOT significant - could be noise'
    print(f'McNemar exact p = {p:.4g}  ({verdict})')


def accuracy_at_budget(frame, budgets=(0.1, 0.2, 0.3, 0.5)):
    ranked = frame.sort_values('confidence', ascending=False)
    rows = []
    for budget in budgets:
        keep = int(round(len(ranked) * (1 - budget)))
        if keep <= 0:
            continue
        auto = ranked.head(keep)
        rows.append({'human_review_budget': budget, 'auto_routed': keep,
                     'accuracy_on_auto': round(accuracy_score(
                         auto['true_label'], auto['predicted_label']), 4)})
    return pd.DataFrame(rows)

budget = accuracy_at_budget(jev).merge(
    accuracy_at_budget(mb), on=['human_review_budget', 'auto_routed'],
    suffixes=('_jev', '_mb'))
print('\nAccuracy at fixed human-review budget:')
print(budget.to_string(index=False))


def ece(frame, n_bins=10):
    bins = pd.cut(frame['confidence'].clip(0, 1),
                  bins=[i / n_bins for i in range(n_bins + 1)], include_lowest=True)
    total, score = len(frame), 0.0
    for _, group in frame.groupby(bins, observed=True):
        if not group.empty:
            score += len(group) / total * abs(group['correct'].mean() - group['confidence'].mean())
    return round(score, 4)

print(f"\nCalibration error (lower is better)")
print(f"  Jev ECE:        {ece(jev)}")
print(f"  ModernBERT ECE: {ece(mb)}")

OUT_DIR = Path('/content/drive/MyDrive/opspilot/jev_benchmark')
OUT_DIR.mkdir(parents=True, exist_ok=True)
jev.to_csv(OUT_DIR / 'jev_predictions.csv', index=False)
mb.to_csv(OUT_DIR / 'modernbert_predictions.csv', index=False)
summary.to_csv(OUT_DIR / 'summary.csv', index=False)
budget.to_csv(OUT_DIR / 'accuracy_at_budget.csv', index=False)
print(f'\nSaved 4 files to {OUT_DIR}')